[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nrao/astrohack/blob/v0.10.1/docs/tutorial_vla.ipynb)

![astrohack](../_media/astrohack_logo.png)

# VLA Holography Tutorial

### Important External Information

- #### xarray Official Documentation ([docs](https://docs.xarray.dev/en/stable/)).
- #### Dask Official Documentation ([docs](https://www.dask.org/)).
- #### zarr Official Documentation ([docs](https://zarr.readthedocs.io/en/stable/))

In [ ]:
import os

try:
    import astrohack

    print("AstroHACK version", astrohack.__version__, "already installed.")
except ImportError as e:
    print(e)
    print("Installing AstroHACK")

    os.system("pip install astrohack")

    import astrohack

    print("astrohack version", astrohack.__version__, " installed.")

## Download Tutorial Data

In [ ]:
import toolviper

toolviper.utils.data.download(file="ea25_cal_small_after_fixed.split.ms", folder="data")

## Holography Data File API

As part of the `astroHACK` API a set of functions to allow users to easily open on disk holography files has been provided. Each function takes an `astroHACK` holography file name as an argument and returns an object related to the given file type, i.e. holog, image, panel, point. Each object allows the user to access data via dictionary keys with values consisting of the relevant holography dataset. Each object also provides a `summary()` helper function to list available keys for each file. An example call for each file type is show below and the API documentation for all data-io functions can be found [here](https://astrohack.readthedocs.io/en/latest/_api/autoapi/astrohack/dio/index.html).

```python
from astrohack import open_holog
from astrohack import open_image
from astrohack import open_panel
from astrohack import open_pointing

holog_data = open_holog('./data/ea25_cal_small_spw1_4_60_ea04_after.holog.zarr')
image_data = open_image('./data/ea25_cal_small_spw1_4_60_ea04_after.image.zarr')
panel_data = open_panel('./data/ea25_cal_small_spw1_4_60_ea04_after.panel.zarr')
pointing_data = open_pointing('./data/ea25_cal_small_spw1_4_60_ea04_after.point.zarr')
```

## Setup Dask Local Cluster

The local Dask client handles scheduling and worker managment for the parallelization. The user has the option of choosing the number of cores and memory allocations for each worker howerver, we recommend a minimum of 8Gb per core with standard settings.


A significant amount of information related to the client and scheduling can be found using the [Dask Dashboard](https://docs.dask.org/en/stable/dashboard.html). This is a built-in dashboard native to Dask and allows the user to monitor the workers during processing. This is especially useful for profilling. For those that are interested in working soley within Jupyterlab a dashboard extension is availabe for [Jupyterlab](https://github.com/dask/dask-labextension#dask-jupyterlab-extension).

![dashboard](../_media/dashboard.png)

In [ ]:
from toolviper.dask.client import local_client

parallel = False

if parallel:
    client = local_client(cores=4, memory_limit="4GB")
    print(client)
else:
    client = None

### Holography processing
## Extract Holog

The extraction and restructuring of the holography data is done using the `extract_holog` function. This function is similar in function to the `UVHOL` task in AIPS. 
The holography data that is extracted can be set using the compound dictionary *holog_obs_description*: *mapping*, *scan*, and *antenna* id. A detailed description of the structure of the *holog_obs_description* dictionary can be found in the documentation [here](https://astrohack.readthedocs.io/en/latest/_api/autoapi/astrohack/extract_holog/index.html). The `extract_holog` can automatically generate the *holog_obs_description* by inspecting the pointing table. 

Inline information on the input parameters can also be gotten using `help(extract_holog)` in the cell.

In [ ]:
from astrohack import extract_pointing, extract_holog

extract_pointing(
    ms_name="data/ea25_cal_small_after_fixed.split.ms",
    point_name="data/ea25_cal_small_after_fixed.split.point.zarr",
    parallel=parallel,
    overwrite=True,
)

holog_mds = extract_holog(
    ms_name="data/ea25_cal_small_after_fixed.split.ms",
    point_name="data/ea25_cal_small_after_fixed.split.point.zarr",
    data_column="CORRECTED_DATA",
    parallel=parallel,
    overwrite=True,
)

`extract_pointing` creates a file for holding the extracted pointing information in the form of `<point_name>.point.zarr`.

___point_name.point.zarr:___ <span style="color:red"> The pointing zarr file contains position and pointing information extracted from the pointing table of the input measurement set. In addition, the antenna and mapping scan information is listed for each antenna. The pointing object is structured in a single depth data tree with the key being the antenna id and the value being the pointing dataset. </span>

```
point_mds = 
{
   ant_0: point_ds,
            ⋮
   ant_n: point_ds
}
```


`extract_holog` creates a file for holding the extracted holography data as `<holog_name>.holog.zarr`. In addition, a holography data object is returned. This is the same holography data object returned by the hologrphy data API above. The `holog_mds` object is a python dict containing the extracted holography data found in `.holog.zarr` but with extended functionality such as providing a summary of the run infomation in table form. Below for each `DDI` we can see the available `scan` and `antenna` information.

___holog_name.holog.zarr:___ <span style="color:red"> The holog zarr file contains ungridded data extracted from the pointing and main tables in the measurement set. The holog file includes the directional, visibility and weight information recorded on a shared time axis; a time resampling is done because the sampling rates of the pointing and main tables are different. In addition, the metadata such as sampled parallactic data (beginning, middle and end of scan) and l and m extents is recorded in the file attributes. The holog file structure is a data tree, with `ant` -> `ddi` -> `map` as the keys for each depth. </span>

```
holog_mds = 
{
   ant_0:{
          ddi_0:{
                 map_0: holog_ds,
                          ⋮
                 map_n: holog_ds
                },
              ⋮
          ddi_p: …
         },
       ⋮
   ant_m: …
}

```

An example of the holog dataset object is show below.

In [ ]:
holog_mds["ant_ea25"]["ddi_0"]["map_0"]

In this case, there is only one selection in the holography file as seen in the summary. Using the available keys we can see an overview of the Dask dataset structure. In addition, the numpy arrays for the data are accessed by calling `values` on a given dataset variable. For instance accessing the data for the `DIRECTIONAL_COSINES` below would be simply
```
>> holog_mds['ddi_0']['map_0']['ant_ea25'].DIRECTIONAL_COSINES.values
>> array([[-0.00433549, -0.0027946 ],
       [-0.00870191, -0.00682571],
       [-0.00965634, -0.00908509],
       ...,
       [ 0.00966373,  0.00957556],
       [ 0.00966267,  0.00957601],
       [ 0.00965895,  0.00956941]])

>> holog_mds['ddi_0']['map_0']['ant_ea25'].DIRECTIONAL_COSINES.values.shape
>> (9145, 2)

```
where the dimension are given in the mds output for each data variable (in this case `(time, lm)`). A more in-depth overview of how to interact with Dask dataset can be found [here](https://tutorial.dask.org/).

A summary of the available key values can be obtained using the summary convenience function and a summary of the observation characteristics can be accessed with the observation_summary function.

In [ ]:
holog_mds.summary()
print(f"\n\n{100*'*'}\n\n")
holog_mds.observation_summary(
    "data/ea25_cal_small_after_fixed.split.holog.zarr.summary.txt"
)

## Holog

The `holog` function processes the holography data and produces a holog image file on disk with the suffix, `.image.zarr`. This function is a direct replacement for the task `HOLOG` in AIPS. It is required that the user provide the `grid_size` and `cell_size` when processing holography data. The `grid_size` defines the number of `l x m`  points used to when doing the gridding. The `cell_size` defines the value in arseconds of each grid spacing. More in-depth parameter information can be found in readthedocs [here](https://astrohack.readthedocs.io/en/latest/_api/autoapi/astrohack/holog/index.html).

Inline information on the input paramters can also be gotten using `help(holog)` in the cell.

In [ ]:
import numpy as np

from astrohack import holog

cell_size = np.array([-0.0006442, 0.0006442])  # arcseconds
grid_size = np.array([31, 31])  # pixels

image_mds = holog(
    holog_name="data/ea25_cal_small_after_fixed.split.holog.zarr",
    overwrite=True,
    phase_fit_engine="perturbations",
    to_stokes=True,
    parallel=parallel,
)

___image_name.image.zarr:___ <span style="color:red"> The image zarr file contains gridded image data the beam, extracted aperture and the amplitude and phase components. It also contains all the relevant coordinate information. The image file structure is a 2 level data tree with keys in the `ant` -> `ddi` order with the leaves consisting of the image dataset. </span>

```
image_mds = 
{
   ant_0:{
          ddi_0: image_ds,
                 ⋮               
          ddi_m: image_ds
         },
       ⋮
   ant_n: …
}

```


An example of the image dataset object is show below.

In [ ]:
image_mds["ant_ea25"]["ddi_0"]

A summary of the available key values can be obtained using the summary convenience function.

In [ ]:
image_mds.summary()

Each of the holography output files is a compound dictionary with respect to the run parameters and contains a xarray Dataset, this means that the holography files have access to all native xarray functionality. The user can use their favorite plotting package to visualize the data or use xarray's internal functions to do simple filtering and plotting.

In [ ]:
image_mds["ant_ea25"]["ddi_0"].CORRECTED_PHASE.isel(chan=0, pol=0).plot()

## Panel

The `panel` function takes the place of and expands the `PANEL` AIPS function to processes the image information and derives adjustements to the dish panels. This produces a file on disk of format `.panel.zarr` containing information on corrections, residuals and screw adjustments. As an added bonus the `panel` function has a helper function to convert aips data to astrohack format and process it using the `aips_holog_to_astrohack` function. For a full description of the operation and arguments of the `panel` function see [docs](https://astrohack.readthedocs.io/en/latest/_api/autoapi/astrohack/panel/index.html).

In [ ]:
from astrohack import panel

panel_model = "rigid"

panel_mds = panel(
    image_name="data/ea25_cal_small_after_fixed.split.image.zarr",
    panel_model=panel_model,
    panel_margins=0.2,
    clip_type="relative",
    clip_level=0.2,
    parallel=parallel,
    overwrite=True,
)

___panel_name.panel.zarr:___ <span style="color:red"> The panel zarr file contains process information regarding the per panel screw corrections as well as residuals, masks and phase corrections used to produce them. The panel file structure is a 2 level data tree with keys in the `ant` -> `ddi` order with the leaves consisting of the panel dataset.</span>

```
panel_mds = 
{
   ant_0:{
          ddi_0: panel_ds,
                 ⋮               
          ddi_m: panel_ds
         },
       ⋮
   ant_n: …

```

An example of the panel dataset object is show below.

In [ ]:
panel_mds["ant_ea25"]["ddi_0"]

A summary of the available key values can be obtained using the summary convenience function. Additionally, a final summary of the observation characteristics can be accessed through the observation_summary method.

In [ ]:
panel_mds.summary()
print(f"\n\n{100*'*'}\n\n")
panel_mds.observation_summary(
    "data/ea25_cal_small_after_fixed.split.panel.zarr.summary.txt"
)

## Additional Functions

The `panel_mds` object provides two helper functions for the user to export or investigate the results of the `panel` function.
- `export_screws()`: This method exports the screw and panel adjustements from the panel output file.
- `plot_antennas()`: This method plots one of three diagnostics plots from the panel output file data. The plots types are: deviation, phase and ancillary.

Examples usage for each helper functions are given below and more detailed documentation can be found in the [visualization tutorial](https://astrohack.readthedocs.io/en/latest/visualization_tutorial.html).

In [ ]:
export_folder = "exports"

panel_mds.export_screws(
    destination=export_folder,
    ant="ea25",
    ddi=0,
    unit="mm",
    threshold=0.5,  # Threshold in mm for significant adjustments
    display=True,
)

In [ ]:
with open(export_folder + "/panel_screws_ant_ea25_ddi_0.txt", "r") as file:
    for _ in range(30):
        print(file.readline()[:-1])

In [ ]:
panel_mds.plot_antennas(
    destination=export_folder,
    ant="ea25",
    ddi=0,
    plot_type="deviation",
    plot_screws=False,
    dpi=300,
    parallel=False,
    display=True,
)

In [ ]:
if parallel:
    client.close()